# TME ML Non supervisé

In [ ]:
import pandas as pd
import sklearn
import numpy as np
import matplotlib.pyplot as plt

# K-Means et Geographical clustering

## 1- Data preparation

<div class="alert alert-block alert-info">
For this section, we will use the points of interest (POI) data of Paris. The objective is to spatially characterize the urban space of Paris based on the POIs present in a region. Clustering the regions thus highlights similar areas and the reasons for their similarity (by analyzing the prototype of each cluster).

The following lines allow you to draw the districts of Paris from the <code>districts-paris.csv</code> file and to load the POIs (from the <code>poi-paris.csv</code> file).
</div>


In [ ]:
table_poi = pd.read_csv('data/poi-paris.csv')
districts=pd.read_csv('data/districts-paris.csv', sep = ',')
districts['bounds'] = districts['bounds'].apply(eval)
#  Draw the districts
def draw_districts():
    for x in districts['bounds']:
        plt.plot([c[0] for c in x], [c[1] for c in x], color = 'black')

In [ ]:
plt.scatter(table_poi["longitude"],table_poi['latitude'],s=1)
draw_districts()

### 2 - Discrétisation manuelle de l'espace

<div class="alert alert-block alert-info">
We cannot use the natural discretization of the districts because it is too coarse and groups together sub-regions of very different types. One initial approach to obtain interesting sub-regions in Paris is to manually discretize using a grid. We will assume a regular grid of size $N$.

Thus, for a GPS coordinate $(long,lat)$, we need to know which grid cell $(i,j)$ it corresponds to. For a grid of size $N \times N$, the formula is as follows: 
$$i=\lfloor N*\frac{(longitude-lomin)}{(lomax-lomin)}\rfloor$$ 
and 
$$j = \lfloor N*\frac{(latitude - lamin)}{(lamax-lamin)} \rfloor,$$
with $\lfloor x\rfloor$ denoting the integer part of $x$, $lamin,lamax$ the minimum and maximum latitudes, and $lomin,lomax$ the minimum and maximum longitudes.

Each grid cell will be described by the distribution of the types of POIs present in the corresponding area. There are 12 types of POIs, so each cell is described by a 12-dimensional vector, where each dimension indicates the probability of the type associated with that dimension in the region.
    
Compute the description associated with each cell. 

Run a k-means algorithm on it (using <code>KMeans</code> from scikit-learn, where <code>n_clusters</code> sets the number of clusters; the <code>fit()</code> method performs the clustering; the <code>predict()</code> method obtains the cluster associated with each example; and the <code>fit_predict()</code> method does both at the same time).

Also, visualize the centroids of the clusters. Vary the number of clusters and compare the results.

The following code graphically represents the computed clustering (with <code>get_clust(i,j)</code> returning the cluster number associated with cell $(i,j)$) and plots the centroids (from a KMeans stored in the variable <code>km</code>).
</div>


In [ ]:
from matplotlib.patches import Rectangle
_ = plt.scatter(table_poi["longitude"],table_poi['latitude'],s=0.1)
draw_districts()
for i in range(N):
    for j in range(N):
        x = lomin + (lomax - lomin) * i / N
        y = lamin + (lamax - lamin) * j / N
        c = tab20.colors[get_clust(i,j)]
        _ = plt.gca().add_patch(Rectangle((x, y), (lomax - lomin) / N, (lamax - lamin) / N, color = c, alpha = 0.5))


In [ ]:
fig, axs = plt.subplots(len(km.cluster_centers_)//2, 2, sharey=True,sharex=True)
for i,k in enumerate(km.cluster_centers_):
    axs.flatten()[i].bar(range(len(k)),k,color=tab20.colors[i])
    axs.flatten()[i].set_xticks(range(len(k)))
    axs.flatten()[i].set_xticklabels(types,rotation="vertical")

<div class="alert alert-block alert-info">
The model's elbow score corresponds to the model's <em>inertia_</em> attribute. Plot the elbow curve as a function of the number of clusters. What is the optimal number of clusters according to this curve?
</div>


## 3 -  K-Means pour la discrétisation automatique de l'espace

<div class="alert alert-block alert-info">
To obtain more relevant clusters, the discretization of space must be done in a less naive way. One idea is also to use the $K$-means algorithm for this, by clustering the GPS coordinates of the POIs (which is also called a <em>quantization</em>). 
</br>
Calculate this spatial clustering and assign each POI a spatial cluster number (note: a grid of size $10\times 10$ is "equivalent" to having 100 clusters).

Graphically represent the clusters thus obtained using the following lines of code (<code>get_cluster_spatial(i)</code> returns the cluster of the $i$-th POI, and the KMeans model is stored in the variable <code>km_spatial</code>).

Perform clustering on this new discretization and compare your results.

Observe the correlation between the types in the initial data. Can you propose a method to improve the results?
</div>


In [ ]:
draw_districts()
plt.scatter(table_poi['longitude'],table_poi['latitude'],color = [tab20.colors[get_cluster_spatial(i) % 20] for i in range(len(table_poi))], s = 0.5)
for i in range(K_GEO):
    plt.plot(km_spatial.cluster_centers_[i, 0], km_spatial.cluster_centers_[i, 1], marker = "*", color = "black", markersize = 5)